# Forecasting time series with skforecast-ai

## Introduction

## Libraries

Libraries used in this document.

In [15]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent.parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
c:\Users\Joaquin\Documents\GitHub\skforecast-ai
0.23.0


In [16]:
# Data processing
# ==============================================================================
import numpy as np
import pandas as pd
from skforecast.datasets import fetch_dataset

# Plots
# ==============================================================================
from skforecast.plot import set_dark_theme
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
plt.style.use('seaborn-v0_8-darkgrid')

# Modelling and Forecasting
# ==============================================================================
import skforecast
import skforecast_ai
import lightgbm
from skforecast_ai import ForecastingAssistant
from skforecast.model_selection import TimeSeriesFold

# Warnings configuration
# ==============================================================================
import warnings
warnings.filterwarnings('once')

color = '\033[1m\033[38;5;208m' 
print(f"{color}Version skforecast_ai: {skforecast_ai.__version__}")
print(f"{color}Version skforecast: {skforecast.__version__}")
print(f"{color}Version lightgbm: {lightgbm.__version__}")

Version skforecast_ai: 0.1.0
Version skforecast: 0.23.0
Version lightgbm: 4.6.0


## Data

The data in this document represent the hourly usage of the bike share system in the city of Washington, D.C. during the years 2011 and 2012. In addition to the number of users per hour, information about weather conditions and holidays is available. The original data was obtained from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset) and has been previously cleaned ([code](https://github.com/JoaquinAmatRodrigo/Estadistica-machine-learning-python/blob/master/code/prepare_bike_sharing_dataset.ipynb)) applying the following modifications:

+ Renamed columns with more descriptive names.

+ Renamed categories of the weather variables. The category of `heavy_rain` has been combined with that of `rain`.

+ Denormalized temperature, humidity and wind variables.

+ `date_time` variable created and set as index.

+ Imputed missing values by forward filling.

In [17]:
# Downloading data
# ==============================================================================
data = fetch_dataset('bike_sharing', raw=True)
data = data[['date_time', 'users', 'holiday', 'weather', 'temp']]
data['date_time'] = pd.to_datetime(data['date_time'], format='%Y-%m-%d %H:%M:%S')
data = data.set_index('date_time')
data = data.asfreq('h')
data = data.sort_index()
data.head()

╭───────────────────────────────── bike_sharing ──────────────────────────────────╮
│ Description:                                                                    │
│ Hourly usage of the bike share system in the city of Washington D.C. during the │
│ years 2011 and 2012. In addition to the number of users per hour, information   │
│ about weather conditions and holidays is available.                             │
│                                                                                 │
│ Source:                                                                         │
│ Fanaee-T,Hadi. (2013). Bike Sharing Dataset. UCI Machine Learning Repository.   │
│ https://doi.org/10.24432/C5W894.                                                │
│                                                                                 │
│ URL:                                                                            │
│ https://raw.githubusercontent.com/skforecast/skforecast-                        │
│ datasets/main/data/bike_sharing_dataset_clean.csv                               │
│                                                                                 │
│ Shape: 17544 rows x 12 columns                                                  │
╰─────────────────────────────────────────────────────────────────────────────────╯

,users,holiday,weather,temp
date_time,,,,
2011-01-01 00:00:00,16.0,0.0,clear,9.84
2011-01-01 01:00:00,40.0,0.0,clear,9.02
2011-01-01 02:00:00,32.0,0.0,clear,9.02
2011-01-01 03:00:00,13.0,0.0,clear,9.84
2011-01-01 04:00:00,1.0,0.0,clear,9.84


In [18]:
assistant = ForecastingAssistant()

In [6]:
profile = assistant.profile(
             data        = data,
             target      = "users",
             date_column = "date_time",
         )
profile

              Dataset Profile              
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property       ┃ Value                  ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Format         │ single                 │
├────────────────┼────────────────────────┤
│ Series         │ 1                      │
├────────────────┼────────────────────────┤
│ Observations   │ 17544                  │
├────────────────┼────────────────────────┤
│ Frequency      │ h                      │
├────────────────┼────────────────────────┤
│ Target         │ users                  │
├────────────────┼────────────────────────┤
│ Exog columns   │ holiday, weather, temp │
├────────────────┼────────────────────────┤
│ Missing target │ none                   │
└────────────────┴────────────────────────┘

                                             Recommendation                                             
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property              ┃ Value                                                                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Task type             │ single_series                                                                │
├───────────────────────┼──────────────────────────────────────────────────────────────────────────────┤
│ Forecaster            │ ForecasterRecursive                                                          │
├───────────────────────┼──────────────────────────────────────────────────────────────────────────────┤
│ Forecaster candidates │ ForecasterRecursive, ForecasterDirect, ForecasterFoundation, ForecasterStats │
├───────────────────────┼──────────────────────────────────────────────────────────────────────────────┤
│ Estimator             │ LGBMRegressor                                                                │
├───────────────────────┼──────────────────────────────────────────────────────────────────────────────┤
│ Estimator candidates  │ LGBMRegressor, XGBRegressor, Ridge                                           │
└───────────────────────┴──────────────────────────────────────────────────────────────────────────────┘

╭────────────────────────────────────────────────── Explanation ──────────────────────────────────────────────────╮
│ A single-series ML forecaster (ForecasterRecursive) is recommended. Data: 17544 observations, 'h' frequency.    │
│ Alternative forecasters: ['ForecasterDirect', 'ForecasterFoundation', 'ForecasterStats']. Estimator:            │
│ LGBMRegressor. A gradient boosting model is preferred for a dataset of this size (17544 observations).          │
│ Alternative estimators: ['XGBRegressor', 'Ridge']. 3 exogenous variables (1 categorical) available as           │
│ predictors.                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [19]:
result = assistant.forecast(
             data        = data,
             target      = "users",
             date_column = "date_time",
             steps       = 36,
         )

In [21]:
# Forecasted values for the next 12 steps
display(result.predictions.head(3))

# Evaluation metrics: MAE, MSE, MASE per series
display(result.metrics)

,pred
2012-08-07 19:00:00,602.275744
2012-08-07 20:00:00,460.290671
2012-08-07 21:00:00,326.293737


,series,MAE,MSE,MASE,MAPE
0,users,21.750571,774.745201,0.366468,0.175919


In [22]:
result

              Dataset Profile              
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property       ┃ Value                  ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Format         │ single                 │
├────────────────┼────────────────────────┤
│ Series         │ 1                      │
├────────────────┼────────────────────────┤
│ Observations   │ 17544                  │
├────────────────┼────────────────────────┤
│ Frequency      │ h                      │
├────────────────┼────────────────────────┤
│ Target         │ users                  │
├────────────────┼────────────────────────┤
│ Exog columns   │ holiday, weather, temp │
├────────────────┼────────────────────────┤
│ Missing target │ none                   │
└────────────────┴────────────────────────┘

                                             Recommendation                                             
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property              ┃ Value                                                                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Task type             │ single_series                                                                │
├───────────────────────┼──────────────────────────────────────────────────────────────────────────────┤
│ Forecaster            │ ForecasterRecursive                                                          │
├───────────────────────┼──────────────────────────────────────────────────────────────────────────────┤
│ Forecaster candidates │ ForecasterRecursive, ForecasterDirect, ForecasterFoundation, ForecasterStats │
├───────────────────────┼──────────────────────────────────────────────────────────────────────────────┤
│ Estimator             │ LGBMRegressor                                                                │
├───────────────────────┼──────────────────────────────────────────────────────────────────────────────┤
│ Estimator candidates  │ LGBMRegressor, XGBRegressor, Ridge                                           │
└───────────────────────┴──────────────────────────────────────────────────────────────────────────────┘

╭────────────────────────────────────────────────── Explanation ──────────────────────────────────────────────────╮
│ A single-series ML forecaster (ForecasterRecursive) is recommended. Data: 17544 observations, 'h' frequency.    │
│ Alternative forecasters: ['ForecasterDirect', 'ForecasterFoundation', 'ForecasterStats']. Estimator:            │
│ LGBMRegressor. A gradient boosting model is preferred for a dataset of this size (17544 observations).          │
│ Alternative estimators: ['XGBRegressor', 'Ridge']. 3 exogenous variables (1 categorical) available as           │
│ predictors.                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
                                                   Forecast Plan                                                   
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property        ┃ Value                                                                                         ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Forecaster      │ ForecasterRecursive                                                                           │
├─────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────┤
│ Estimator       │ LGBMRegressor                                                                                 │
├─────────────────┼─────────────────────────────────────────────────────────────────────────────────────────────

In [28]:
result.show_code()

╭──────────────────────────────────────────────── Generated code ─────────────────────────────────────────────────╮
│ import pandas as pd                                                                                             │
│ from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error             │
│ from skforecast.metrics import mean_absolute_scaled_error                                                       │
│ from lightgbm import LGBMRegressor                                                                              │
│ from skforecast.preprocessing import RollingFeatures, CalendarFeatures                                          │
│ from skforecast.recursive import ForecasterRecursive                                                            │
│                                                                                                                 │
│ # Load data                                                                                                     │
│ data = pd.read_csv('data.csv', index_col=0, parse_dates=True)                                                   │
│                                                                                                                 │
│ data = data.asfreq('h')                                                                                         │
│ data = data.sort_index()                                                                                        │
│                                                                                                                 │
│ # Train/test split                                                                                              │
│ end_train = '2012-08-07 18:00:00'  # 80% of data, adjust to change the split point                              │
│ data_train = data.loc[:end_train]                                                                               │
│ data_test  = data.loc[data.index > end_train]                                                                   │
│ exog_features = ['holiday', 'weather', 'temp']                                                                  │
│                                                                                                                 │
│ print(                                                                                                          │
│     f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"               │
│ )                                                                                                               │
│ print(                                                                                                          │
│     f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"                  │
│ )                                                                                                               │
│                                                                                                                 │
│ window_features = RollingFeatures(                                                                              │
│     stats        = ['mean', 'std', 'mean', 'mean'],                                                             │
│     window_sizes = [3, 3, 24, 168],                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
│ calendar_features = CalendarFeatures(                                                  

In [12]:
# Split train-validation-test
# ==============================================================================
end_train = '2012-04-30 23:59:00'
end_validation = '2012-08-31 23:59:00'
data_train = data.loc[: end_train, :]
data_val   = data.loc[end_train:end_validation, :]
data_test  = data.loc[end_validation:, :]

print(f"Dates train      : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})")
print(f"Dates validation : {data_val.index.min()} --- {data_val.index.max()}  (n={len(data_val)})")
print(f"Dates test       : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})")

Dates train      : 2011-01-01 00:00:00 --- 2012-04-30 23:00:00  (n=11664)
Dates validation : 2012-05-01 00:00:00 --- 2012-08-31 23:00:00  (n=2952)
Dates test       : 2012-09-01 00:00:00 --- 2012-12-31 23:00:00  (n=2928)


In [13]:
cv = TimeSeriesFold(steps = 36, initial_train_size = end_validation)
result = assistant.backtest(
             data        = data,
             target      = "users",
             date_column = "date_time",
             cv          = cv,
         )

Information of folds
--------------------
Number of observations used for initial training: 14616
Number of observations used for backtesting: 2928
    Number of folds: 82
    Number skipped folds: 0 
    Number of steps per fold: 36
    Number of steps to exclude between last observed data (last window) and predictions (gap): 0
    Last fold only includes 12 observations.

Fold: 0
    Training:   2011-01-01 00:00:00 -- 2012-08-31 23:00:00  (n=14616)
    Validation: 2012-09-01 00:00:00 -- 2012-09-02 11:00:00  (n=36)
Fold: 1
    Training:   No training in this fold
    Validation: 2012-09-02 12:00:00 -- 2012-09-03 23:00:00  (n=36)
Fold: 2
    Training:   No training in this fold
    Validation: 2012-09-04 00:00:00 -- 2012-09-05 11:00:00  (n=36)
Fold: 3
    Training:   No training in this fold
    Validation: 2012-09-05 12:00:00 -- 2012-09-06 23:00:00  (n=36)
Fold: 4
    Training:   No training in this fold
    Validation: 2012-09-07 00:00:00 -- 2012-09-08 11:00:00  (n=36)
Fold: 5
    Tr

100%|██████████| 82/82 [00:01<00:00, 57.41it/s]


In [14]:
result

╭────────────────────────────────────────────────── Explanation ──────────────────────────────────────────────────╮
│ Initial training up to 2012-08-31 23:59:00, fixed window, no refit, 36-step horizon, 82 folds. Results —        │
│ mean_absolute_error: 46.5843, mean_squared_error: 5443.1711, mean_absolute_scaled_error: 0.7540,                │
│ mean_absolute_percentage_error: 0.4703.                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
       Cross-Validation Configuration       
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Parameter          ┃               Value ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ steps              │                  36 │
├────────────────────┼─────────────────────┤
│ initial_train_size │ 2012-08-31 23:59:00 │
├────────────────────┼─────────────────────┤
│ refit              │               False │
├────────────────────┼─────────────────────┤
│ fixed_train_size   │                True │
├────────────────────┼─────────────────────┤
│ gap                │                   0 │
├────────────────────┼─────────────────────┤
│ fold_stride        │                  36 │
├────────────────────┼─────────────────────┤
│ differentiation    │                None │
└────────────────────┴─────────────────────┘
                                             Backtest Metrics                                             
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ mean_absolute_error ┃ mean_squared_error ┃ mean_absolute_scaled_error ┃ mean_absolute_percentage_error ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│             46.5843 │          5443.1711 │                     0.7540 │                         0.4703 │
└─────────────────────┴────────────────────┴────────────────────────────┴────────────────────────────────┘
╭─────────────────────────────────────── Backtest Predictions (2928 rows) ────────────────────────────────────────╮
│                      fold        pred                                                                           │
│ 2012-09-01 00:00:00     0  125.532963                                                                           │
│ 2012-09-01 01:00:00     0   99.116233                                                                           │
│ 2012-09-01 02:00:00     0   73.377862                                                                           │
│ 2012-09-01 03:00:00     0   34.199683                                                                           │
│ 2012-09-01 04:00:00     0   12.101884                                                                           │
│   ...                                                                                                           │
│ 2012-12-31 19:00:00  81  149.788718                                                                             │
│ 2012-12-31 20:00:00  81  106.500335                                                                             │
│ 2012-12-31 21:00:00  81   67.891513                                                                             │
│ 2012-12-31 22:00:00  81   48.436067                                                                             │
│ 2012-12-31 23:00:00  81   32.888916                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
              Dataset Profile              
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Property       ┃ Value                  ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Format         │ single                 │
├────────────────┼────────────────────────┤
│ Series         │ 1                      │
├────────────────┼──────────────────